# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - LSTM Binary Model (Depreciated)

### Written by Krupam Patel
### Written in Python 3.11.14

###### This file contains the code for the Binary LSTM for upwelling and onshore flags prediction.

In [117]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    classification_report,
    confusion_matrix
)
import os
import pandas as pd
import numpy as np

In [118]:
# Train one model at a time.
TARGET = "Onshore"
#TARGET = "upwelling_flag"

WINDOW = 24
THRESHOLD = 0.5

# Start without weighting
USE_CLASS_WEIGHTS = False

In [119]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CLEAN.csv') 

In [120]:
cleancsv["Date"] = (
    pd.to_datetime(cleancsv["Date"], errors="coerce")
    + pd.to_timedelta(cleancsv["Hr"], unit="h")
)

cleancsv = cleancsv.drop(columns="Hr")
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

if cleancsv["Date"].duplicated().any():
    raise ValueError("Resolve duplicate timestamps before training.")

# Use the same months across models for comparison.
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv.loc[summer_mask].reset_index(drop=True)

In [121]:
# Validate the selected binary target BEFORE conversion.
target_values = cleancsv[TARGET].to_numpy()

if not np.isin(target_values, [0, 1]).all():
    raise ValueError(
        f"{TARGET} must contain only 0 and 1. "
        f"Found: {np.unique(target_values)}"
    )

output_data = target_values.astype(np.int32)

# Historical flags may be inputs: windows end before the target hour.
feature_cols = [
    column for column in cleancsv.columns
    if column != "Date"
]

dataset = cleancsv[feature_cols].to_numpy(dtype=np.float32)

if not np.isfinite(dataset).all():
    raise ValueError("Inputs contain missing or infinite values.")

In [122]:
# Length
n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

if not WINDOW < train_end < test_start < n:
    raise ValueError("Not enough observations for these splits.")

In [123]:
scaler_x = StandardScaler()
scaler_x.fit(dataset[:train_end])
scaledx = scaler_x.transform(dataset)

# Require consecutive hours through the target timestamp.
hourly_step = (
    cleancsv["Date"].diff() == pd.Timedelta(hours=1)
)

valid_target = (
    hourly_step
    .rolling(window=WINDOW, min_periods=WINDOW)
    .sum()
    .eq(WINDOW)
)

target_rows = np.flatnonzero(valid_target.to_numpy())

if len(target_rows) == 0:
    raise ValueError("No complete 24-hour forecast windows found.")

In [124]:
X = np.stack([
    scaledx[i - WINDOW:i]
    for i in target_rows
])

# No target scaling or class-index subtraction.
y = output_data[target_rows].astype(np.float32).reshape(-1, 1)

train_mask = target_rows < train_end
val_mask = (
    (target_rows >= train_end)
    & (target_rows < test_start)
)
test_mask = target_rows >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

test_rows = target_rows[test_mask]

# Inspect class availability in each period.
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    if labels.size == 0:
        raise ValueError(f"{name} split has no valid windows.")

    counts = np.bincount(
        labels.ravel().astype(int),
        minlength=2
    )

    print(f"{name}: zeros={counts[0]}, ones={counts[1]}")

    if np.any(counts == 0):
        print(f"WARNING: {name} contains only one class.")

if np.unique(y_train).size < 2:
    raise ValueError(
        f"Training data for {TARGET} contain only one class. "
        "More data containing both 0 and 1 are needed."
    )

print("Training shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Testing shape:", X_test.shape, y_test.shape)

Training: zeros=1052, ones=671
Validation: zeros=75, ones=144
Testing: zeros=176, ones=42
Training shape: (1723, 24, 13) (1723, 1)
Validation shape: (219, 24, 13) (219, 1)
Testing shape: (218, 24, 13) (218, 1)


In [125]:
class_weight = None

if USE_CLASS_WEIGHTS:
    counts = np.bincount(
        y_train.ravel().astype(int),
        minlength=2
    )

    class_weight = {
        0: float(len(y_train) / (2 * counts[0])),
        1: float(len(y_train) / (2 * counts[1]))
    }

    print("Class weights:", class_weight)

In [126]:
print("X_train:", X_train.shape)   # (samples, 24, 13)
print("y_train:", y_train.shape)   # (samples, 1)
print("First 10 labels:", y_train[:10].ravel())

X_train: (1723, 24, 13)
y_train: (1723, 1)
First 10 labels: [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


In [127]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

model = keras.models.Sequential([
    keras.layers.Input(shape=(WINDOW, len(feature_cols))),
    keras.layers.LSTM(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

In [128]:
#Put all the layers together
model.summary()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,977 (27.25 KB)

 Trainable params: 6,977 (27.25 KB)

 Non-trainable params: 0 (0.00 B)

In [129]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    shuffle=False,
    class_weight=class_weight,
    callbacks=[early_stopping]
)

Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6686 - loss: 0.6390 - precision: 0.6157 - recall: 0.3964 - val_accuracy: 0.7489 - val_loss: 0.6396 - val_precision: 0.9009 - val_recall: 0.6944
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7731 - loss: 0.5505 - precision: 0.8302 - recall: 0.5246 - val_accuracy: 0.7626 - val_loss: 0.5794 - val_precision: 0.8833 - val_recall: 0.7361
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7789 - loss: 0.4965 - precision: 0.8021 - recall: 0.5738 - val_accuracy: 0.7763 - val_loss: 0.5285 - val_precision: 0.8682 - val_recall: 0.7778
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7806 - loss: 0.4639 - precision: 0.7867 - recall: 0.5991 - val_accuracy: 0.7808 - val_loss: 0.5146 - val_precision: 0.8636 - val_recall: 0.7917
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7876 - loss: 0.4437 - precision: 0.7883 - recall: 0.6215 - val_accuracy: 0.7991 - val_loss: 0.5053 

In [130]:
prediction_prob = model.predict(X_test).ravel()

prediction_flag = (
    prediction_prob >= THRESHOLD
).astype(np.int32)

y_test_true = y_test.ravel().astype(np.int32)

# Persistence: next hour has the previous hour's flag.
naive_prediction = output_data[test_rows - 1]

# Additional baseline: always predict no event.
zero_prediction = np.zeros_like(y_test_true)

print(f"\nTarget: {TARGET}")
print(f"Threshold: {THRESHOLD}")

for name, predicted in [
    ("LSTM", prediction_flag),
    ("Persistence", naive_prediction),
    ("Always zero", zero_prediction)
]:
    mae = mean_absolute_error(y_test_true, predicted)

    print(f"\n{name} flag MAE: {mae:.4f}")
    print(f"{name} incorrect forecasts: {mae * 100:.2f}%")

    print(classification_report(
        y_test_true,
        predicted,
        labels=[0, 1],
        target_names=["No event", "Event"],
        zero_division=0
    ))

print("LSTM confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(
    y_test_true,
    prediction_flag,
    labels=[0, 1]
))

if np.unique(y_test_true).size < 2:
    print(
        "WARNING: The test set contains only one class. cannot assess performance on both event and non-event hours."
    )

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

Target: Onshore
Threshold: 0.5

LSTM flag MAE: 0.1422
LSTM incorrect forecasts: 14.22%
              precision    recall  f1-score   support

    No event       0.91      0.92      0.91       176
       Event       0.64      0.60      0.62        42

    accuracy                           0.86       218
   macro avg       0.77      0.76      0.76       218
weighted avg       0.85      0.86      0.86       218


Persistence flag MAE: 0.1330
Persistence incorrect forecasts: 13.30%
              precision    recall  f1-score   support

    No event       0.92      0.92      0.92       176
       Event       0.66      0.64      0.65        42

    accuracy                           0.87       218
   macro avg       0.79      0.78      0.78       218
weighted avg       0.87      0.87      0.87       218


Always zero flag MAE: 0.1927
Always zero incorrect forecasts: 19.27%
              precision    recall  f1-score   support

    No event       0.81  

In [131]:
test_df = pd.DataFrame({
    "Date": cleancsv["Date"].iloc[test_rows].to_numpy(),
    f"observed_{TARGET}": y_test_true,
    f"predicted_{TARGET}_prob": prediction_prob,
    f"predicted_{TARGET}_flag": prediction_flag,
    f"predicted_{TARGET}_naive": naive_prediction
})

os.makedirs("CSV/predictions", exist_ok=True)

output_path = f"CSV/predictions/{TARGET}_lstm_predictions.csv"

test_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: CSV/predictions/Onshore_lstm_predictions.csv
